2D卷积

In [15]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class Myconv2d(nn.Module):
    def __init__(self, input_channels, output_channels, kernels, padding=0, stride=1, use_bias=True):
        super(Myconv2d,self).__init__()
        self.input_channels = input_channels
        self.output_channels = output_channels
        self.kernel_size = kernels if isinstance(kernels,tuple) else (kernels,kernels)
        self.padding = padding
        self.stride = stride
        self.use_bias = use_bias

        #设置权重和偏置,简单使用torch,rand进行初始化
        self.weights = nn.Parameter(torch.rand(self.output_channels, self.input_channels, *self.kernel_size))
        self.bias = nn.Parameter(torch.rand(self.output_channels))

    def forward(self,x):
        bs, c, h, w = x.shape
        output_h = (h-self.kernel_size[0]+2*self.padding)//self.stride+1
        output_w = (w-self.kernel_size[1]+2*self.padding)//self.stride+1

        x_conv_unfolded = F.unfold(x, kernel_size=self.kernel_size, padding=self.padding, stride=self.stride)#(bs,一个卷积核的大小,L),L=output_h*output_w
        x_conv = torch.matmul(self.weights.reshape(self.output_channels,-1), x_conv_unfolded)#（bs,out_channels,L）
        if self.use_bias:
            x_conv = x_conv+self.bias.view(1,-1,1)
        x_output = x_conv.reshape(bs, self.output_channels, output_h, output_w)

        return x_output
# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
image_feature = torch.rand(4, 64, 100, 100)
myconv2d = Myconv2d(64, 128, 3)
out = myconv2d(image_feature)
print(image_feature.shape)
print(out.shape)


"""
torch.Size([4, 64, 100, 100])
torch.Size([4, 128, 98, 98])
"""

torch.Size([4, 64, 100, 100])
torch.Size([4, 128, 98, 98])


'\ntorch.Size([4, 64, 100, 100])\ntorch.Size([4, 128, 98, 98])\n'

2D卷积简单调用

In [19]:
import torch
import torch.nn as nn
import torch.nn.functional as F


in_channels = 1
out_channels = 1
kernel_size = 3
bias = False
batch_size = 1
input_size = [batch_size,in_channels, 4 ,4]

# 普普通通的一个卷积
conv_layer = torch.nn.Conv2d(
    in_channels,
    out_channels,
    kernel_size,
    bias = bias
)

# 随机产生一个input
input = torch.randn(input_size)
output = conv_layer(input)
print(conv_layer.weight)
print(conv_layer.weight.size())


Parameter containing:
tensor([[[[ 0.3220,  0.0136, -0.0196],
          [ 0.0881, -0.0621,  0.2521],
          [-0.3248, -0.0958,  0.2414]]]], requires_grad=True)
torch.Size([1, 1, 3, 3])


Mnist 数据集
卷积，全连接，池化

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

# 1. 数据加载与预处理
transform = transforms.Compose([
    transforms.ToTensor(),  # 转为张量
    transforms.Normalize((0.5,), (0.5,))  # 归一化到 [-1, 1]
])

# 加载 MNIST 数据集
train_dataset = datasets.MNIST(root='./data', train=True, transform=transform, download=True)
test_dataset = datasets.MNIST(root='./data', train=False, transform=transform, download=True)

train_loader = torch.utils.data.DataLoader(dataset=train_dataset, batch_size=64, shuffle=True)
test_loader = torch.utils.data.DataLoader(dataset=test_dataset, batch_size=64, shuffle=False)

# 2. 定义 CNN 模型
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        # 定义卷积层
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1)  # 输入1通道，输出32通道
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)  # 输入32通道，输出64通道
        # 定义全连接层
        self.fc1 = nn.Linear(64 * 7 * 7, 128)  # 展平后输入到全连接层
        self.fc2 = nn.Linear(128, 10)  # 10 个类别

    def forward(self, x):
        x = F.relu(self.conv1(x))  # 第一层卷积 + ReLU
        x = F.max_pool2d(x, 2)     # 最大池化
        x = F.relu(self.conv2(x))  # 第二层卷积 + ReLU
        x = F.max_pool2d(x, 2)     # 最大池化
        x = x.view(-1, 64 * 7 * 7) # 展平
        x = F.relu(self.fc1(x))    # 全连接层 + ReLU
        x = self.fc2(x)            # 最后一层输出
        return x

# 创建模型实例
model = SimpleCNN()

# 3. 定义损失函数与优化器
criterion = nn.CrossEntropyLoss()  # 多分类交叉熵损失
optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)

# 4. 模型训练
num_epochs = 5
model.train()  # 设置模型为训练模式

for epoch in range(num_epochs):
    total_loss = 0
    for images, labels in train_loader:
        outputs = model(images)  # 前向传播
        loss = criterion(outputs, labels)  # 计算损失

        optimizer.zero_grad()  # 清空梯度
        loss.backward()  # 反向传播
        optimizer.step()  # 更新参数

        total_loss += loss.item()

    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {total_loss / len(train_loader):.4f}")

# 5. 模型测试
model.eval()  # 设置模型为评估模式
correct = 0
total = 0

with torch.no_grad():  # 关闭梯度计算
    for images, labels in test_loader:
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f"Test Accuracy: {accuracy:.2f}%")

# 6. 可视化测试结果
dataiter = iter(test_loader)
images, labels = next(dataiter)
outputs = model(images)
_, predictions = torch.max(outputs, 1)

fig, axes = plt.subplots(1, 6, figsize=(12, 4))
for i in range(6):
    axes[i].imshow(images[i][0], cmap='gray')
    axes[i].set_title(f"Label: {labels[i]}\nPred: {predictions[i]}")
    axes[i].axis('off')
plt.show()
'''
100.0%
100.0%
100.0%
100.0%
Epoch [1/5], Loss: 0.2321
Epoch [2/5], Loss: 0.0546
Epoch [3/5], Loss: 0.0382
Epoch [4/5], Loss: 0.0283
Epoch [5/5], Loss: 0.0237
Test Accuracy: 98.80%
'''

一个简单的神经网络

In [21]:
import torch.nn as nn
import torch.optim as optim

# 定义一个简单的全连接神经网络
class SimpleNN(nn.Module):
    def __init__(self):
        super(SimpleNN, self).__init__()
        self.fc1 = nn.Linear(2, 2)  # 输入层到隐藏层
        self.fc2 = nn.Linear(2, 1)  # 隐藏层到输出层

    def forward(self, x):
        x = torch.relu(self.fc1(x))  # ReLU 激活函数
        x = self.fc2(x)
        return x

# 创建网络实例
model = SimpleNN()

# 打印模型结构
print(model)

SimpleNN(
  (fc1): Linear(in_features=2, out_features=2, bias=True)
  (fc2): Linear(in_features=2, out_features=1, bias=True)
)


# Deepseek老师教我网络层

H_out = [(H_in + 2×padding[0] - dilation[0]×(kernel_size[0]-1) - 1)/stride[0] + 1]

W_out = [(W_in + 2×padding[1] - dilation[1]×(kernel_size[1]-1) - 1)/stride[1] + 1]

这就是前面第一个conv2d实现的原理 二维卷积

简化版本（当dilation=1时）：

H_out = (H_in + 2×padding - kernel_size)//stride + 1

W_out = (W_in + 2×padding - kernel_size)//stride + 1

In [22]:
import torch.nn as nn

# 基本卷积层
conv1 = nn.Conv2d(
    in_channels=3,      # 输入通道 (RGB图像为3)
    out_channels=64,    # 输出通道 (64个滤波器)
    kernel_size=3,      # 3x3卷积核
    stride=1,           # 步长1
    padding=1           # 填充1，保持尺寸不变
)

# 下采样卷积
conv2 = nn.Conv2d(
    in_channels=64,
    out_channels=128,
    kernel_size=3,
    stride=2,           # 步长2，尺寸减半
    padding=1
)

# 1x1卷积 - 用于调整通道数
conv3 = nn.Conv2d(64, 128, kernel_size=1, stride=1)
print(conv3)

Conv2d(64, 128, kernel_size=(1, 1), stride=(1, 1))


## 全连接层

原理

将输入向量的每个元素与输出向量的每个元素相连接，实现特征的综合和维度变换。

关键参数

in_features：输入向量长度

out_features：输出向量长度

In [24]:
# 基本全连接层
fc1 = nn.Linear(
    in_features=1024,   # 输入维度
    out_features=512    # 输出维度
)

# 分类器输出层
classifier = nn.Linear(512, 10)  # 10个类别的分类

# 使用示例
x = torch.randn(32, 1024)  # [batch_size, features]
output = fc1(x)            # [32, 512]
print(output)

tensor([[ 0.4744, -0.3456,  1.5514,  ..., -0.0370,  0.3277,  0.2893],
        [-0.5618, -0.3397,  0.6341,  ...,  0.0102, -0.6087,  0.8959],
        [-0.0790,  0.9515,  0.4400,  ...,  1.2574, -0.6393, -0.9200],
        ...,
        [ 0.1552, -0.3712, -0.3147,  ..., -0.8761,  0.5444, -0.4495],
        [ 0.2276, -0.7763,  0.4325,  ..., -0.3169, -0.0613,  0.1773],
        [-2.3011, -0.3769, -0.1568,  ...,  0.1839,  0.4126, -0.6168]],
       grad_fn=<AddmmBackward0>)


## 池化层
原理

对局部区域进行下采样，减少数据量，增强特征的平移不变性。

类型
 - 最大池化：取局部区域最大值

 - 平均池化：取局部区域平均值

 - 全局池化：对整个特征图进行池化

In [25]:
# 最大池化
max_pool = nn.MaxPool2d(
    kernel_size=2,      # 2x2池化窗口
    stride=2           # 步长2，尺寸减半
)

# 平均池化
avg_pool = nn.AvgPool2d(
    kernel_size=2,
    stride=2
)

# 全局平均池化 - 将每个通道降为1x1
global_avg_pool = nn.AdaptiveAvgPool2d((1, 1))

# 保持尺寸的池化
same_size_pool = nn.MaxPool2d(
    kernel_size=3,
    stride=1,
    padding=1          # 保持尺寸不变
)

## 全卷积网络
原理与实现

用卷积层替代全连接层，实现任意尺寸输入和空间信息保留

In [26]:
class FullyConvolutionalNet(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()

        # 特征提取部分
        self.features = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(128, 128, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        # 分类头 - 全卷积方式
        self.classifier = nn.Sequential(
            nn.Conv2d(128, 256, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(256, 256, 3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),  # 全局平均池化
            nn.Conv2d(256, num_classes, 1) # 1x1卷积替代全连接
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x.flatten(1)  # 展平为 [batch, num_classes]

## 批归一化层
原理
对每个批次的数据进行归一化，稳定训练过程。

关键参数

num_features：通道数

eps：数值稳定性常数

momentum：运行统计量的动量

affine：是否学习缩放和偏移参数

In [27]:
# 2D批归一化（用于卷积层后）
bn_conv = nn.BatchNorm2d(
    num_features=64,    # 通道数
    eps=1e-5,          # 防止除零
    momentum=0.1,      # 运行均值和方差的动量
    affine=True        # 学习γ和β参数
)

# 1D批归一化（用于全连接层后）
bn_fc = nn.BatchNorm1d(512)

# 在网络中的使用
class CNNWithBN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 64, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(2)

        self.conv2 = nn.Conv2d(64, 128, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(128)

        self.fc = nn.Linear(128*8*8, 10)

    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.pool(x)
        x = self.relu(self.bn2(self.conv2(x)))
        x = self.pool(x)
        x = x.flatten(1)
        x = self.fc(x)
        return x

In [29]:
import torch
import torch.nn as nn

class CompleteCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()

        # 卷积块1
        self.block1 = nn.Sequential(
            nn.Conv2d(3, 64, 3, stride=1, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, 3, stride=1, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2)  # 尺寸减半
        )

        # 卷积块2
        self.block2 = nn.Sequential(
            nn.Conv2d(64, 128, 3, stride=1, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, 3, stride=1, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1))  # 全局平均池化
        )

        # 分类器
        self.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(128, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.block1(x)      # [B, 64, H/2, W/2]
        x = self.block2(x)      # [B, 128, 1, 1]
        x = x.flatten(1)        # [B, 128]
        x = self.classifier(x)  # [B, num_classes]
        return x

# 使用示例
model = CompleteCNN(num_classes=10)
input_tensor = torch.randn(32, 3, 32, 32)  # [batch, channels, height, width]
output = model(input_tensor)                # [32, 10]
print(f"输入尺寸: {input_tensor.shape}")
print(f"输出尺寸: {output.shape}")
print(output)

输入尺寸: torch.Size([32, 3, 32, 32])
输出尺寸: torch.Size([32, 10])
tensor([[ 0.1165, -0.1250,  0.3954, -0.0185,  0.1266,  0.4019,  0.0457,  0.1324,
          0.0339,  0.1286],
        [ 0.3609, -0.1827, -0.0620,  0.0207,  0.1991,  0.4618,  0.0559,  0.0809,
          0.3641, -0.0161],
        [ 0.0938, -0.1523,  0.1280,  0.0189,  0.3308,  0.4800,  0.0646, -0.2851,
          0.2463, -0.1602],
        [ 0.0284, -0.0990,  0.1691,  0.0965,  0.1152,  0.5774,  0.0841,  0.0147,
          0.2745,  0.0257],
        [ 0.1699, -0.0374, -0.1859, -0.0237,  0.1429,  0.2468,  0.1299,  0.0724,
          0.0230, -0.2075],
        [ 0.0461, -0.1231,  0.0284,  0.0978,  0.0778,  0.3756,  0.0872,  0.1210,
          0.3905,  0.1612],
        [ 0.2015,  0.0635, -0.1438,  0.0457,  0.2747,  0.3676, -0.2339, -0.1132,
          0.0989, -0.0084],
        [ 0.1708,  0.0148, -0.1079,  0.0412,  0.1768,  0.4979,  0.0537,  0.2569,
          0.1183, -0.2982],
        [ 0.1471,  0.0297, -0.0197,  0.1351,  0.1777,  0.1804,  0.1